# Elastic Net

### Both penalties, and why you would want both

**Made by Elyes Lounissi** ·
[LinkedIn](https://www.linkedin.com/in/elyes-lounissi/) ·
[pilot.tun@gmail.com](mailto:pilot.tun@gmail.com) ·
[all notebooks](../../CURRICULUM.md)

---

| | |
|---|---|
| **What you will learn** | What mixing L1 and L2 buys, the grouping effect that neither penalty has alone, how to tune two hyperparameters without fooling yourself, and when the extra complexity is not worth it |
| **You should already know** | [Ridge](../04-ridge-regression/) and [Lasso](../05-lasso-regression/) |
| **Dataset** | California Housing, with correlated groups and noise columns planted |
| **Runtime** | Two to three minutes on a laptop CPU |
| **Next** | [02-07 Outlier-resistant regression](../07-outlier-resistant-regression/) |

---

## 1. The idea

The two previous notebooks each ended on a limitation.

[Ridge](../04-ridge-regression/) gave stable coefficients but never removed a
feature — its 30 noise columns all survived with small weights.

[Lasso](../05-lasso-regression/) removed features, but among correlated twins it
kept **one arbitrarily**, and the choice jumped between resamples.

Elastic Net adds both penalties and lets you set the mix:

$$J(w) = \|Xw - y\|^2 + \alpha\Big(\rho \sum_i |w_i| + \frac{1-\rho}{2}\sum_i w_i^2\Big)$$

`alpha` sets the total strength, `l1_ratio` ($\rho$) sets the balance. At
$\rho = 1$ it is exactly Lasso; at $\rho = 0$ exactly Ridge.

The interesting behaviour lives strictly in between, and it is not simply "a bit
of each". Adding even a small L2 term produces a **grouping effect**: correlated
features get kept or dropped *together*, with the weight shared between them.
Neither penalty does that alone.

In [ ]:
import sys
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "toolkit").is_dir())
sys.path.insert(0, str(ROOT))

from toolkit import datasets, style

style.use()
FIG = pathlib.Path("figures")
SEED = 0

from sklearn.preprocessing import StandardScaler

X, y = datasets.california_housing()
rng = np.random.default_rng(SEED)

# Three near-copies of the most useful feature, plus 20 pure noise columns.
X_built = X.copy()
for i in range(3):
    X_built[f"MedInc_copy{i}"] = X["MedInc"] + rng.normal(0, 0.02, len(X))
noise = rng.normal(size=(len(X), 20))
names = list(X_built.columns) + [f"noise_{i}" for i in range(20)]
X_full = np.hstack([X_built.values, noise])
TWINS = ["MedInc", "MedInc_copy0", "MedInc_copy1", "MedInc_copy2"]
twin_index = [names.index(c) for c in TWINS]

print(f"{X_full.shape[0]:,} rows, {X_full.shape[1]} columns")
print(f"  {X.shape[1]} real, 3 near-copies of MedInc, 20 pure noise")

## 2. The grouping effect

The direct test. Fit all three penalties and look at what each does to the four
near-identical income columns.

In [ ]:
from sklearn.linear_model import ElasticNet, Lasso, Ridge

X_scaled = StandardScaler().fit_transform(X_full)

fits = {
    "Ridge (l1_ratio=0)": Ridge(alpha=1.0),
    "Elastic Net (0.5)": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=50000),
    "Elastic Net (0.9)": ElasticNet(alpha=0.01, l1_ratio=0.9, max_iter=50000),
    "Lasso (l1_ratio=1)": Lasso(alpha=0.01, max_iter=50000),
}

rows = []
for label, model in fits.items():
    coefficients = model.fit(X_scaled, y).coef_
    twin_weights = coefficients[twin_index]
    rows.append({
        "model": label,
        **{c: w for c, w in zip(TWINS, twin_weights)},
        "twins kept": int((np.abs(twin_weights) > 1e-8).sum()),
        "twin sum": twin_weights.sum(),
        "noise kept": int((np.abs(coefficients[len(X_built.columns):]) > 1e-8).sum()),
    })
grouping = pd.DataFrame(rows).set_index("model")

fig, ax = plt.subplots(figsize=(9.0, 4.2))
positions = np.arange(len(grouping))
bottom = np.zeros(len(grouping))
for i, twin in enumerate(TWINS):
    ax.bar(positions, grouping[twin], bottom=bottom, width=0.6,
           color=style.PALETTE[i], label=twin)
    bottom += grouping[twin].values
ax.set_xticks(positions, [m.replace(" (", "\n(") for m in grouping.index], fontsize=9)
ax.set_ylabel("weight on each income twin (stacked)")
ax.legend(fontsize=8.5, ncols=2)
style.title(ax, "Lasso hands everything to one twin. Elastic Net shares it out.",
            "four near-identical copies of MedInc, correlation above 0.999")
style.save(fig, FIG / "fig-01-grouping.png")

print(grouping.round(3).to_string())

Read the `twins kept` column. Lasso keeps one of the four and zeroes the rest;
Ridge keeps all four but also keeps every noise column; Elastic Net keeps the
group together **and** discards noise.

The `twin sum` column is the check that they are all solving the same problem —
the total weight given to income is similar in every row. They differ only in how
it is distributed.

### And is the choice stable?

In [ ]:
records = []
for trial in range(12):
    sample = rng.choice(len(X_scaled), 4000, replace=True)
    for label, model in [("lasso", Lasso(alpha=0.01, max_iter=50000)),
                         ("elastic net", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=50000))]:
        coefficients = model.fit(X_scaled[sample], y.values[sample]).coef_
        records.append({"trial": trial, "model": label,
                        **{c: coefficients[i] for c, i in zip(TWINS, twin_index)}})
stability = pd.DataFrame(records)

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.2), sharey=True)
for ax, label in zip(axes, ["lasso", "elastic net"]):
    subset = stability[stability.model == label]
    for i, twin in enumerate(TWINS):
        ax.plot(subset["trial"], subset[twin], marker=style.MARKERS[i],
                color=style.PALETTE[i], label=twin)
    spread = np.mean([subset[t].std() for t in TWINS])
    ax.set_xlabel("bootstrap sample")
    style.title(ax, label.title(), f"mean coefficient spread {spread:.3f}")
axes[0].set_ylabel("weight on each twin")
axes[1].legend(fontsize=8)
style.save(fig, FIG / "fig-02-stability.png")

for label in ["lasso", "elastic net"]:
    subset = stability[stability.model == label]
    print(f"{label:<12}: mean spread across 12 refits = "
          f"{np.mean([subset[t].std() for t in TWINS]):.4f}")

## 3. Tuning two dials at once

Two hyperparameters means a grid, and it means being careful. `ElasticNetCV`
searches `l1_ratio` and `alpha` together using cross-validation, and the search
has to sit **inside** the outer evaluation or the score is optimistic.

In [ ]:
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline

folds = KFold(5, shuffle=True, random_state=SEED)
ratios = [0.1, 0.3, 0.5, 0.7, 0.9, 0.99]
alphas = np.logspace(-4, 0, 25)

surface = np.zeros((len(ratios), len(alphas)))
for i, ratio in enumerate(ratios):
    for j, alpha in enumerate(alphas):
        model = make_pipeline(StandardScaler(),
                              ElasticNet(alpha=alpha, l1_ratio=ratio, max_iter=20000))
        surface[i, j] = -cross_val_score(model, X_full, y, cv=folds,
                                         scoring="neg_root_mean_squared_error",
                                         n_jobs=-1).mean()

fig, ax = plt.subplots(figsize=(8.8, 4.4))
image = ax.imshow(surface, aspect="auto", cmap="Oranges_r", origin="lower")
best_i, best_j = np.unravel_index(np.argmin(surface), surface.shape)
ax.scatter([best_j], [best_i], s=170, marker="*", c=style.INK, zorder=5)
ax.set_xticks(range(0, len(alphas), 4), [f"{a:.4f}" for a in alphas[::4]], rotation=45, fontsize=8)
ax.set_yticks(range(len(ratios)), [str(r) for r in ratios])
ax.set_xlabel("alpha")
ax.set_ylabel("l1_ratio")
ax.grid(False)
fig.colorbar(image, ax=ax, shrink=0.85, label="cross-validated RMSE")
style.title(ax, f"Best: alpha={alphas[best_j]:.4f}, l1_ratio={ratios[best_i]}",
            "the valley is broad, which is good news for tuning")
style.save(fig, FIG / "fig-03-tuning.png")

print(f"best RMSE on the grid : {surface.min():.4f}")
print(f"  at alpha={alphas[best_j]:.5f}, l1_ratio={ratios[best_i]}")
print(f"worst RMSE on the grid: {surface.max():.4f}")

The valley is broad and flat. That is worth noticing, because it means the exact
`l1_ratio` barely matters as long as `alpha` is sensible — the two dials are far
from equally important, and most of the work is done by `alpha`.

## 4. Is any of it worth it?

The honest comparison, on the constructed dataset with correlated twins and
planted noise, which is the situation Elastic Net was designed for.

In [ ]:
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV

contenders = {
    "ordinary least squares": LinearRegression(),
    "ridge (CV)": RidgeCV(alphas=np.logspace(-3, 3, 30)),
    "lasso (CV)": LassoCV(cv=5, max_iter=50000, random_state=SEED),
    "elastic net (CV)": ElasticNetCV(l1_ratio=ratios, cv=5, max_iter=50000,
                                     random_state=SEED, n_jobs=-1),
}

results = []
for label, model in contenders.items():
    pipeline = make_pipeline(StandardScaler(), model)
    score = -cross_val_score(pipeline, X_full, y, cv=folds,
                             scoring="neg_root_mean_squared_error", n_jobs=-1).mean()
    fitted = pipeline.fit(X_full, y)
    coefficients = fitted[-1].coef_
    results.append({
        "model": label,
        "RMSE": score,
        "noise kept": int((np.abs(coefficients[len(X_built.columns):]) > 1e-8).sum()),
        "twins kept": int((np.abs(coefficients[twin_index]) > 1e-8).sum()),
    })
comparison = pd.DataFrame(results).set_index("model")

fig, axes = plt.subplots(1, 2, figsize=(12.4, 3.9))
positions = np.arange(len(comparison))
axes[0].barh(positions, comparison["RMSE"], color=style.NEUTRAL, height=0.6)
axes[0].barh([positions[-1]], [comparison["RMSE"].iloc[-1]], color=style.HIGHLIGHT, height=0.6)
for i, v in enumerate(comparison["RMSE"]):
    axes[0].text(v + 0.003, i, f"{v:.4f}", va="center", fontsize=9, color=style.MUTED)
axes[0].set_yticks(positions, comparison.index, fontsize=9)
axes[0].set_xlim(0.6, 0.85)
axes[0].set_xlabel("cross-validated RMSE")
axes[0].grid(axis="y", visible=False); axes[0].grid(axis="x", visible=True)
style.title(axes[0], "Accuracy", "all four land in the same place")

axes[1].barh(positions, comparison["noise kept"], color=style.NEUTRAL, height=0.6)
axes[1].barh([positions[-1]], [comparison["noise kept"].iloc[-1]], color=style.HIGHLIGHT, height=0.6)
for i, v in enumerate(comparison["noise kept"]):
    axes[1].text(v + 0.3, i, f"{v}", va="center", fontsize=9, color=style.MUTED)
axes[1].set_yticks(positions, ["" for _ in positions])
axes[1].set_xlabel("noise columns kept, of 20")
axes[1].grid(axis="y", visible=False); axes[1].grid(axis="x", visible=True)
style.title(axes[1], "Parsimony", "which is where they actually differ")

style.save(fig, FIG / "fig-04-comparison.png")

print(comparison.round(4).to_string())

## Cheat sheet

| | |
|---|---|
| **Use it when** | You have correlated groups of features **and** want selection. That combination is the entire reason it exists |
| **Prefer Lasso when** | Features are largely independent and you want the simplest possible model |
| **Prefer Ridge when** | You want stability and have no interest in removing features |
| **Scaling needed** | Yes, as for both parents |
| **Main dials** | `alpha` for strength, `l1_ratio` for the mix. `alpha` matters far more |
| **Tuning** | `ElasticNetCV` searches both. Keep it inside the outer cross-validation |
| **Watch out** | Two hyperparameters is twice the chance of tuning on the test set. It also converges slowly — expect to raise `max_iter` |

## What to remember

1. Elastic Net is $\rho$ parts Lasso and $1-\rho$ parts Ridge.
2. The grouping effect is the real reason to use it: correlated features are kept
   or dropped together, with the weight shared.
3. That makes the selected feature set far more stable than Lasso's across resamples.
4. The tuning surface is broad, so `alpha` does most of the work and `l1_ratio`
   tolerates being approximately right.
5. On accuracy alone all three regularisers usually tie. The difference is which
   features survive, and how reliably.

---

**Made by Elyes Lounissi** ·
[LinkedIn](https://www.linkedin.com/in/elyes-lounissi/) ·
[pilot.tun@gmail.com](mailto:pilot.tun@gmail.com)

Next: [02-07 Outlier-resistant regression](../07-outlier-resistant-regression/) ·
Back to [the curriculum](../../CURRICULUM.md)

`#MachineLearning` `#ElasticNet` `#Regularization` `#L1` `#L2` `#FeatureSelection`
`#Python` `#ScikitLearn` `#DataScience` `#MLTutorial` `#Regression`